<a href="https://colab.research.google.com/github/juli0AND/Evaluaci-n-comparativa-de-YOLOv5-y-YOLOv8/blob/main/Grad_CAM_YOLOV8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 -q
!pip install ultralytics==8.0.196 grad-cam roboflow -q
!git clone https://github.com/Spritan/YOLOv8_Explainer
%cd YOLOv8_Explainer
!pip install -e .
import torch
import ultralytics

print("Torch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
%cd /content/YOLOv8_Explainer
!ls
%pip install roboflow thop pandas -q
#IMPORTAR DATASET DE ROBOFLOW
from roboflow import Roboflow
rf = Roboflow(api_key="io4o0d2qhkzDHpqNE3Tb")
project = rf.workspace("deteccion-residuos").project("new-classification-yolov5_nano-pk95c")
version = project.version(1)
dataset = version.download("yolov8")

from google.colab import files
files.upload() #best.pt
from google.colab import files
files.upload() #Sample images of YOLOV5

with open("imagenes_prueba (2).txt", "r") as f:
    imagenes_antiguas = [line.strip() for line in f]
	import os

imagenes_prueba = []

for ruta in imagenes_antiguas:
    nombre = os.path.basename(ruta)
    nueva_ruta = f"{dataset.location}/test/images/{nombre}"
    imagenes_prueba.append(nueva_ruta)
for i in range(20):
    print(imagenes_prueba[i], os.path.exists(imagenes_prueba[i]))
import os

imagenes_prueba = []

base = "/content/YOLOv8_Explainer/NEW-CLASSIFICATION-WASTE-yolo_nano-1"

for ruta in imagenes_antiguas:
    nombre = os.path.basename(ruta)

    for carpeta in ["train", "valid", "test"]:
        nueva = f"{base}/{carpeta}/images/{nombre}"

        if os.path.exists(nueva):
            imagenes_prueba.append(nueva)
            break
class YOLOv8Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.model

    def forward(self, x):
        preds = self.model(x)
        return preds[0]
		wrapped_model = YOLOv8Wrapper(model)
wrapped_model.eval()
target_layers = [wrapped_model.model.model[-2]]
cam = EigenCAM(
    model=wrapped_model,
    target_layers=target_layers
)
with open("imagenes_prueba (2).txt", "r") as f:
    imagenes_antiguas = [line.strip() for line in f]
	import os

base = dataset.location

imagenes_prueba = []

for ruta in imagenes_antiguas:
    nombre = os.path.basename(ruta)
    nueva = f"{base}/test/images/{nombre}"

    if os.path.exists(nueva):
        imagenes_prueba.append(nueva)

import cv2
import numpy as np
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt

os.makedirs('gradcams_v8', exist_ok=True)

for i, img_path in enumerate(imagenes_prueba):

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_float = img.astype(np.float32) / 255.0

    input_tensor = (
        torch.from_numpy(img_float)
        .permute(2,0,1)
        .unsqueeze(0)
        .float()
    )

    grayscale_cam = cam(input_tensor=input_tensor)[0]

    visualization = show_cam_on_image(
        img_float,
        grayscale_cam,
        use_rgb=True
    )

    plt.figure(figsize=(8,8))
    plt.imshow(visualization)
    plt.axis('off')

    plt.savefig(
        f'gradcams_v8/gradcam_v8_{i+1}.png',
        dpi=1000,
        bbox_inches='tight',
        pad_inches=0
    )

    plt.close()

    print(f'Imagen {i+1}/20 guardada')